In [15]:
import vertexai

client = vertexai.Client(project='rajesh-more-cwx-internal', location='us-central1')
remote_agent_engine = client.agent_engines.get(name='projects/519243981219/locations/us-central1/reasoningEngines/8395400198720847872')

async def test_agent():
    async for event in remote_agent_engine.async_stream_query(
        message="hello",
        user_id="test_user"
    ):
        print(event)

# In Jupyter, just run:
await test_agent()

{'content': {'parts': [{'thought_signature': 'CpgBAePx_178_xj0Qy3RfN7w5NkaDsGlFVU0_mTC6ex7pXcICaof4m2j9lEZVterAepYByqkVUx-1mT-wN-KukyvACGHUOrqpfIJKi8ToDL4xbBLu4X0kK1wKIpZiJ-LCJsKM8c5ildJGNb_b66MqPLfVfkoSB8kp2DNwrvnvqcWLjBPYEh3BxUF1eGQZ78O1KzwtKuVjGjJmcI=', 'function_call': {'id': 'adk-2a0c3c2d-59af-4e0a-a824-bd6d1f072f3b', 'args': {'query': 'hello'}, 'name': 'greeting'}}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 3, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 3}], 'prompt_token_count': 2930, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 2930}], 'thoughts_token_count': 26, 'total_token_count': 2959, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -3.2397209803263345, 'invocation_id': 'e-26132533-8d13-41a9-8d59-9bd90356262f', 'author': 'Train_Agent', 'actions': {'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}, 'long_running_tool_ids': [], 'id': '0e

In [7]:
def _extract_from_dict_event(event: dict) -> str:
    """Extract text from dictionary-style events (AgentEngine format)."""
    # Handle AgentEngine response format
    if "parts" in event and "role" in event:
        # Only extract text from model responses, skip function calls/responses
        if event.get("role") == "model":
            parts = event.get("parts", [])
            text_parts = []

            for part in parts:
                # Extract text content, skip function calls
                if isinstance(part, dict) and "text" in part:
                    text_parts.append(part["text"])

            return "".join(text_parts) if text_parts else None
        return None

    # Handle other dict formats
    content = event.get("content", {})
    if isinstance(content, str):
        return content

    parts = content.get("parts", [])
    if not parts:
        return None

    text_parts = []
    for part in parts:
        if isinstance(part, dict) and "text" in part:
            text_parts.append(part["text"])

    return "".join(text_parts) if text_parts else None


Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x15eb2aad0>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x15eb28e10>


{'content': {'parts': [{'thought_signature': 'CpECAePx_1659jXPSUt4NGd2_m26uLHbEN0QXfxy2kVA85jBGDMqNDTxgfyh36INibD004wB8vDjYn9u3nzMUOrWKWFq-Dx7zNXIHsfGL8_joRHhLpTOrTuTQRGJ4UJjNS1vL_7V95yqKQpW4mqEafVJeg0x5B6uGcj9Er62jFMeJL4LRHa-PARbQBSBkpFWM-CEDhYjmS5XAFX9auNTHz_Zw23eZWi7v3nN4t07Cq8qbkK1hOuh-yGaUwFS5eGCO-_EbJyySPCbK76wXxWriE40TCJvUE4zwwnN9kPwDhYrMIU6iI9w_xvYkmjQNeQn9C1uE64dTaqb6z8v3-dei53ardGUMgJ5be71XXL9XsZmJIXH', 'function_call': {'id': 'adk-0b4a7a41-57b5-4a78-ad70-15a0d6d89a80', 'args': {'query': 'Hello'}, 'name': 'greeting'}}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 3, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 3}], 'prompt_token_count': 2935, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 2935}], 'thoughts_token_count': 45, 'total_token_count': 2983, 'traffic_type': 'ON_DEMAND'}, 'avg_logprobs': -4.040157636006673, 'invocation_id': 'e-7dca9258-be07-4e3b-9e8a-eb7ef699a206', 'author': 'Train_Agent',

In [27]:
import asyncio # Make sure asyncio is imported

async def stream_agent_response(query: str, user_id: str):
    """
    Asynchronously streams an agent's response and prints formatted events.
    """
    print(f"🚀 Starting agent query for '{user_id}': '{query}'\n" + "-"*30)
    responses = []
    try:
        # Start the asynchronous stream
        async for event in remote_agent_engine.async_stream_query(
            message=query,
            user_id=user_id
        ):
            
            # --- NOTE: Adjust event attributes (like 'type', 'delta', 'tool_name') ---
            # --- based on your specific agent framework (e.g., LlamaIndex, LangChain) ---

            # Default to 'unknown' if 'type' attribute doesn't exist
            event_type = getattr(event, 'type', 'unknown')

            if event_type == 'text_chunk':
                # Print text chunks as they arrive, without newlines
                text_delta = getattr(event, 'delta', '')
                print(text_delta, end="", flush=True)
            
            elif event_type == 'tool_call':
                # Show when the agent decides to use a tool
                tool_name = getattr(event, 'tool_name', 'unknown_tool')
                print(f"\n\n🔧 Using tool: {tool_name}...")
            
            elif event_type == 'tool_result':
                # Show the result from the tool (truncate for readability)
                result = str(getattr(event, 'result', ''))
                print(f"\n✅ Tool result: {result[:100]}...")
            
            elif event_type == 'final_response':
                # Streaming is done (this event may or may not contain the full text)
                pass # We've already printed chunks, so just let it finish
            
            else:
                # Print any other event types for debugging
                text = _extract_from_dict_event(event)
                responses.append(text)
                print(f"\n[DEBUG] Event: {text}")
        
    except Exception as e:
        print(f"\n\n🚨 An error occurred during the stream: {e}")
    
    finally:
        # Always print a footer when the stream finishes or fails
        print("\n" + "-"*30 + "\n🏁 Agent stream finished.")
        return "".join([x for x in responses if x!=None])


In [28]:

 responses = await stream_agent_response(
     query="Show me trains from DELHI JN. (DLI) to MUMBAI CENTRAL (MMCT)",
     user_id="test_user"
 )

🚀 Starting agent query for 'test_user': 'Show me trains from DELHI JN. (DLI) to MUMBAI CENTRAL (MMCT)'
------------------------------

[DEBUG] Event: None

[DEBUG] Event: None

[DEBUG] Event: Here are the trains found from Delhi Junction (DLI) to Mumbai Central (MMCT) for today:

Trains from Delhi Jn (DLI) to Mumbai Central (MMCT)

| Train No | Name | Departure | Arrival | Duration | Days | Classes |
|----------|-------------------|-----------|---------|----------|--------|-----------------|
| 12172 | HW LTT AC SF | NZM 22:40 | LTT 21:30 | 22h 50m | TF | 1A,2A,3A |
| 12472 | SWARAJ EXPRESS | SZM 20:42 | BDTS 16:10 | 19h 28m | TWFS | 1A,2A,3A,3E,SL |
| 22210 | MMCT DURONTO | NZM 22:20 | MMCT 15:50 | 17h 30m | TS | 1A,2A,3A,3E,2S |
| 22222 | CSMT RAJDHANI | NZM 16:55 | CSMT 11:15 | 18h 20m | SMTWTFS | 1A,2A,3A |
| 19020 | HW BDTS EXP | NZM 18:55 | BDTS 22:35 | 27h 40m | SMTWTFS | 1A,2A,3A,3E,SL |
| 09004 | MMCT FESTIVLSPL | DSJ 10:54 | MMCT 10:30 | 23h 36m | SMTWTFS | 3A |
| 01494 | PUNE

In [29]:
print(responses)

Here are the trains found from Delhi Junction (DLI) to Mumbai Central (MMCT) for today:

Trains from Delhi Jn (DLI) to Mumbai Central (MMCT)

| Train No | Name | Departure | Arrival | Duration | Days | Classes |
|----------|-------------------|-----------|---------|----------|--------|-----------------|
| 12172 | HW LTT AC SF | NZM 22:40 | LTT 21:30 | 22h 50m | TF | 1A,2A,3A |
| 12472 | SWARAJ EXPRESS | SZM 20:42 | BDTS 16:10 | 19h 28m | TWFS | 1A,2A,3A,3E,SL |
| 22210 | MMCT DURONTO | NZM 22:20 | MMCT 15:50 | 17h 30m | TS | 1A,2A,3A,3E,2S |
| 22222 | CSMT RAJDHANI | NZM 16:55 | CSMT 11:15 | 18h 20m | SMTWTFS | 1A,2A,3A |
| 19020 | HW BDTS EXP | NZM 18:55 | BDTS 22:35 | 27h 40m | SMTWTFS | 1A,2A,3A,3E,SL |
| 09004 | MMCT FESTIVLSPL | DSJ 10:54 | MMCT 10:30 | 23h 36m | SMTWTFS | 3A |
| 01494 | PUNE AC FEST SPL | NZM 21:25 | KYN 21:23 | 23h 58m | TF | 3A |
| 12618 | MNGLA LKSDP | NZM 05:35 | PNVL 06:30 | 24h 55m | SMTWTFS | 2A,3A,3E,SL |
| 12926 | PASCHIM EXPRESS | SZM 15:40 | BDTS 14:30